<a href="https://colab.research.google.com/github/alireza-asl/startup-prediction/blob/main/startupPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#upload csv data
from google.colab import files
uploaded = files.upload()

In [ ]:
#Import packadges
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('startup data.csv')
df.head()


In [ ]:
# Check the shape of the dataset
df.shape

In [ ]:
# Info about columns and data types
df.info()

Data Cleaning

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
#calculate the percentage of missing values for each column
null_percent = (df.isnull().sum() / len(df)) * 100
print(null_percent)

In [ ]:
#drop unnecessary features
df.drop(columns=['Unnamed: 0', 'Unnamed: 6', 'latitude', 'longitude', 'zip_code', 'id', 'object_id', 'name'], inplace = True)

In [ ]:
#recheck missing values after dropping columns
print((df.isnull().sum() / len(df)) * 100)

In [ ]:
#create a new column: is_closed (1 = closed, 0 = still active)
df['is_closed'] = df['closed_at'].notnull().astype(int)

In [ ]:
df['is_closed'].value_counts()

In [ ]:
df['state_code'].value_counts()

In [ ]:
#checking wheather 'state_code' == 'state_code.1'? maybe we drop one of them
(df['state_code'] == df['state_code.1']).all()

In [ ]:
#check the case/s where state_code != state_code.1
dif = df.loc[df['state_code'] != df['state_code.1']]
dif

In [ ]:
df = df.drop('state_code.1', axis = 1)

In [ ]:
#check categories in 'state_code'
df['state_code'].value_counts().to_frame()

In [ ]:
#keep the first 5 categories,More than 80% of categories of 'state_code' are from the first 5 categories
state_mapping = {'CA': 'CA', 'NY': 'NY', 'MA': 'MA', 'TX': 'TX', 'WA': 'WA'}
df['State'] = df['state_code'].map(state_mapping).fillna('other')

In [ ]:
# Visualization
state_counts = df['State'].value_counts()

plt.figure(figsize=(7, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
myexplode = [0, 0.2, 0, 0, 0, 0]
plt.pie(state_counts, labels=state_counts.index, autopct='%1.1f%%', startangle=140, colors=colors, explode = myexplode)
plt.title('Distribution of State', fontsize = 18)
plt.show()

In [ ]:
#drop unnecessary features
df.drop(columns=['state_code', 'is_CA', 'is_NY', 'is_MA', 'is_TX', 'is_otherstate'], inplace = True)

In [ ]:
#check values of 'category_code' column
df['category_code'].value_counts(normalize=True).to_frame(name='Proportion')
df['category_code'].value_counts().to_frame().assign(Proportion=lambda x: x / x.sum())

In [ ]:
#we take top 10 categories from 'category_code' cloumn
mapping = {
    'software': 'software', 'web': 'web', 'mobile': 'mobile',
    'enterprise': 'enterprise', 'advertising': 'advertising',
    'games_video': 'games_video', 'semiconductor': 'semiconductor',
    'network_hosting': 'network_hosting', 'biotech': 'biotech', 'hardware': 'hardware'
}

df['category'] = df['category_code'].map(mapping).fillna('other')

In [ ]:
#visualize category with bar Chart
category_counts = df['category'].value_counts()

plt.figure(figsize=(10, 5))
plt.bar(category_counts.index, category_counts.values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'])
plt.xlabel('Category')
plt.ylabel('Count')
plt.title('Category Distribution', fontsize = 18)
plt.xticks(rotation=45)
plt.show()

In [ ]:
df['category'].value_counts(normalize=True).to_frame(name='Proportion')
df['category'].value_counts().to_frame().assign(Proportion=lambda x: x / x.sum())

In [ ]:
#propotion of other categories is 25% --> not too big, Selected categories represents about 75% --> not bad
df.drop(columns=['category_code', 'is_software', 'is_web', 'is_mobile', 'is_enterprise', 'is_advertising',
                 'is_gamesvideo', 'is_ecommerce', 'is_biotech', 'is_consulting', 'is_othercategory'], inplace = True)

In [ ]:
for column in df.columns:
    print(f"{column}: Number of unique values {df[column].nunique()}")

In [ ]:
#Check if 'labels' is the same feature as 'status'?
df['status'].value_counts()

In [ ]:
df['status'] = np.where(df['status']=='acquired',1,0)

In [ ]:
(df['labels'] == df['status']).all()

In [ ]:
#drop 'labels'
df = df.drop(['labels'], axis = 1)

In [ ]:
prop_df = df.groupby('status').size().reset_index(name = 'counts')
prop_df['proportions'] = prop_df['counts']/prop_df['counts'].sum()

In [ ]:
import matplotlib.pyplot as plt

#define labels and colors
labels = ['Acquired', 'Closed']
colors = ['#1f77b4', '#ff7f0e']
myexplode = [0, 0.1]

#create pie chart
plt.figure(figsize=(7, 6))
plt.pie(prop_df['proportions'], labels=labels, autopct='%1.1f%%', colors=colors, startangle=140, explode = myexplode)
plt.title('Distribution of Status of the Startup', fontsize = 18)
plt.show()

In [ ]:
#convert the 'founded_at' column from a date to just extract and use year
df['founded_at'] = pd.to_datetime(df['founded_at'])
df['founded_year'] = df['founded_at'].dt.year

In [ ]:
#checking if 'is_closed' is matching with the target 'status'?
df_check_closed = df.loc[(df['is_closed']!='0') & (df['status'] == 1)]
df_check_closed.style.set_properties(**{'background-color': 'yellow'}, subset=['is_closed','status'])

In [ ]:
df = df.drop(['founded_at'], axis = 1)

In [ ]:
oldest_year = df['founded_year'].min()
newest_year = df['founded_year'].max()

print(f"Oldest founded year: {oldest_year}")
print(f"Newest founded year: {newest_year}")

In [ ]:
df_prop = df['founded_year'].value_counts(normalize=True).reset_index()
df_prop.columns = ['founded_year', 'proportions']

df_prop = df_prop.sort_values('founded_year')

In [ ]:
#visualize 'founded_year' using both line + area Graph together
fig, ax = plt.subplots(figsize=(20, 8))

#line plot
ax.plot(df_prop['founded_year'], df_prop['proportions'], color='blue', marker='o', label='Proportion Line')

#area plot (filled under the curve)
ax.fill_between(df_prop['founded_year'], df_prop['proportions'], color='skyblue', alpha=0.5, label='Proportion Area')

plt.title('Line + Area Graph: Distribution of Startups by Year (Acquired/Not)')
plt.xlabel('Founded Year')
plt.ylabel('Proportion')
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
#filling missing values of age_first_milestone_year and age_last_milestone_year
df['age_first_milestone_year'] = df['age_first_milestone_year'].fillna(value="0")
df['age_last_milestone_year'] = df['age_last_milestone_year'].fillna(value="0")

In [ ]:
#check null values in features again
null_columns = df.columns[df.isnull().any()]
print(null_columns)

 Data Preprocessing:

In [ ]:
#check duplicates
print("Duplicate Rows:")
df[df.duplicated()]

In [ ]:
#drop duplicates
df = df.drop_duplicates()

In [ ]:
#dealing with negative values
cols = ["age_first_funding_year", "age_last_funding_year",
        "age_first_milestone_year", "age_last_milestone_year"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = np.where(df[col] < 0, 0, df[col])

In [ ]:
#check if there are any NaN values exist
print(df[cols].isna().sum())

In [ ]:
#general method to count outliers
def count_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers

In [ ]:
for col in df.select_dtypes(include=np.number).columns:
    print("{:<15} {:>6}".format(col, len(count_outliers(df, col))))

In [ ]:
df['has_roundC'].value_counts()

In [ ]:
for column in df.columns:
    print(f"{column}: Number of unique values {df[column].nunique()}")

In [ ]:
out_cols = ['age_first_funding_year','age_last_funding_year','age_first_milestone_year','age_last_milestone_year',
            'relationships', 'funding_rounds', 'funding_total_usd', 'avg_participants']
plt.figure(figsize=(15, 7))

for i, col in enumerate(out_cols, 1):
    plt.subplot(1, len(out_cols), i)
    sns.boxplot(y=df[col], color='lightslategrey', orient='v')

plt.tight_layout()

In [ ]:
#apply Log transformation
df['funding_total_usd'] = np.log1p(df['funding_total_usd'])
df['relationships'] = np.log1p(df['relationships'])
df['avg_participants'] = np.log1p(df['avg_participants'])

In [ ]:
#remove outliers using IQR
def remove_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

df = remove_outliers_iqr(df, 'funding_rounds')
df = remove_outliers_iqr(df, 'age_first_funding_year')
df = remove_outliers_iqr(df, 'age_last_funding_year')
df = remove_outliers_iqr(df, 'age_first_milestone_year')
df = remove_outliers_iqr(df, 'age_last_milestone_year')

In [ ]:
df.head()

In [ ]:
out_cols = ['age_first_funding_year','age_last_funding_year','age_first_milestone_year','age_last_milestone_year',
            'relationships', 'funding_rounds', 'funding_total_usd', 'avg_participants']
plt.figure(figsize=(15, 7))

for i, col in enumerate(out_cols, 1):
    plt.subplot(1, len(out_cols), i)
    sns.boxplot(y=df[col], color='lightslategrey', orient='v')

plt.tight_layout()

In [ ]:
#encode categorical variables
df = pd.get_dummies(df, columns=['State', 'category'], drop_first=True)

In [ ]:
#label encoding
le = LabelEncoder()
df['city'] = le.fit_transform(df['city'])

In [ ]:
df.head()

In [ ]:
df.drop(columns=['first_funding_at', 'last_funding_at'], inplace = True)

Feature Engineering

In [ ]:
df['has_Investor'] = np.where((df['has_VC'] == 1) | (df['has_angel'] == 1), 1, 0)

In [ ]:
df['has_RoundABCD'] = np.where((df['has_roundA'] == 1) | (df['has_roundB'] == 1) | (df['has_roundC'] == 1) | (df['has_roundD'] == 1), 1, 0)

In [ ]:
df['has_Seed'] = np.where((df['has_RoundABCD'] == 0) & (df['has_Investor'] == 1), 1, 0)

In [ ]:
df['invalid_startup'] = np.where((df['has_RoundABCD'] == 0) & (df['has_VC'] == 0) & (df['has_angel'] == 0), 1, 0)

In [ ]:
df.head()

modeling

In [ ]:
X = df.drop(columns=['status'])
y = df['status']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
scaler = StandardScaler()
X_train_resampled = scaler.fit_transform(X_train_resampled)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from lightgbm import LGBMClassifier

In [ ]:
models = {
    "LGBMClassifier": LGBMClassifier(),
    "AdaBoostClassifier": AdaBoostClassifier(),
    "RandomForestClassifier": RandomForestClassifier(),
}

model training and evaluation

In [ ]:
#train & evaluate each model
for name, model in models.items():
    print(f"\n Training {name}...\n")

    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)

    #classification Report
    print(f"Classification Report for {name}:\n")
    print(classification_report(y_test, y_pred))

    #confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 0', 'Class 1'], yticklabels=['Class 0', 'Class 1'])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {name}")
    plt.show()
plt.close()


In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
# Scaling data before LogisticRegression model
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

In [ ]:
#random Forest
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
#support Vector Machine (SVM)
from sklearn.svm import SVC
model = SVC()
model.fit(X_train, y_train)

In [ ]:
#k-Nearest Neighbors (KNN)
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))